In [ ]:
# 1. 필수 라이브러리 설치 (약 1분 소요)
!pip install -q insightface onnxruntime-gpu
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python-headless pillow numpy lpips
!pip install -q flask-ngrok pyngrok flask_cors

print("✅ 라이브러리 설치 완료")

In [ ]:
# 2. GitHub에서 필터 엔진(nofake_filter.py) 다운로드
# 주의: 아래 주소의 '저장소이름'을 실제 리포지토리 이름으로 수정하세요.
!wget https://raw.githubusercontent.com/DKU-Opensource7/저장소이름/main/nofake_filter.py -O nofake_filter.py

# 잘 받아졌는지 확인
import os
if os.path.exists("nofake_filter.py"):
    print("✅ 필터 파일(nofake_filter.py) 다운로드 성공!")
else:
    print("❌ 파일 다운로드 실패. 주소를 확인해주세요.")

In [ ]:
# 3. 웹 서버 실행 (ngrok)
import os
from flask import Flask, request, send_file
from pyngrok import ngrok
from flask_cors import CORS
import cv2
import numpy as np

# 우리가 만든 필터 함수 불러오기
from nofake_filter import apply_deepfake_protection

app = Flask(__name__)
CORS(app) # 모든 외부 접속 허용

# ==================================================
# [필수] 본인의 ngrok 토큰을 아래 따옴표 안에 넣으세요!
# ==================================================
NGROK_TOKEN = "여기에_토큰을_넣으세요"
ngrok.set_auth_token(NGROK_TOKEN)

@app.route('/upload', methods=['POST'])
def upload_file():
    # 1. 파일 확인
    if 'file' not in request.files:
        return "No file uploaded", 400

    file = request.files['file']

    # 2. 강도(Strength) 확인 (기본값: medium)
    # 웹페이지에서 'high', 'medium', 'low' 중 하나를 보냄
    strength = request.form.get('strength', 'medium')

    # 3. 임시 저장
    input_path = "temp_input.jpg"
    output_path = "temp_output.jpg"
    file.save(input_path)

    try:
        # 4. 필터 적용 (강도 전달)
        apply_deepfake_protection(input_path, output_path, strength=strength)

        # 5. 결과 반환
        return send_file(output_path, mimetype='image/jpeg')

    except Exception as e:
        print(f"Error: {e}")
        return f"Server Error: {str(e)}", 500

# 서버 포트 열기
public_url = ngrok.connect(5000).public_url
print(f"\n🚀 [서버 가동 중] 웹 팀에게 이 주소를 주세요: {public_url}/upload")

app.run(port=5000)